In [ ]:
pip install langchain langchain-core langchain-community openai chromadb tiktoken langchain-chroma langchain-openai ragas -qU

In [1]:
import os
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.schema import Document
import json

In [6]:
# 경로 설정
DIRECTORY_PATH = r"Raw_DB"
PERSIST_DIRECTORY = r"Raw_DB\vector_store\contents"
COLLECTION_NAME = "contents"
EMBEDDING_MODEL_NAME = "text-embedding-ada-002"

In [7]:
# JSON 데이터 불러오기
with open("Raw_DB/total.json", "r", encoding="utf-8") as f:
    total = json.load(f)

# Document 객체로 변환
documents = []
for idx in range(len(total)):
    content = total[idx]
    merged_text = f"id: {content["id"]}, type: {content["type"]}, platform: {content["platform"]}, title: {content["title"]}, status: {content["status"]}, thumbnail: {content["thumbnail"]}, genre: {content['genre']}, views: {content["views"]}, rating: {content["rating"]}, like: {content["like"]}, description: {content["description"]}, keyword: {content["keywords"]}, author: {content['author']}, illustrator: {content['illustrator']}, original: {content['original']}, age_rating: {content['age_rating']}, price: {content['price']}"
    if isinstance(content["price"], str):
        price = content["price"]
    else:
        price = ", ".join(content["price"])
    if isinstance(content["keywords"], str):
        keywords = content["keywords"]
    else:
        keywords = ", ".join(content["keywords"])
    documents.append(
        Document(
            page_content=merged_text,
            metadata={
                "id": content["id"],
                "type": content["type"],
                "platform": content["platform"],
                "title": content["title"],
                "status": content["status"],
                "thumbnail": content["thumbnail"],
                "genre": content["genre"],
                "views": content["views"],
                "rating": content["rating"],
                "like": content["like"],
                "description": content["description"],
                "keywords": keywords,
                "author": content["author"],
                "illustrator": content["illustrator"],
                "original": content["original"],
                "age_rating": content["age_rating"],
                "price": price,
                "url": content["url"],
                "episode": content["episode"],
            },
        )
    )

In [ ]:
from pprint import pprint

pprint(documents)

In [14]:
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME)

if os.path.exists(PERSIST_DIRECTORY):
    vector_store = Chroma(
        persist_directory=PERSIST_DIRECTORY,
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
    )
else:
    vector_store = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        collection_name=COLLECTION_NAME,
        persist_directory=PERSIST_DIRECTORY,
    )

vector_store.persist()

print(
    "=== 내용 확인 ======================================================================================================================================"
)
try:
    results = vector_store.similarity_search("화산귀환")
    print(results[0] if results else "검색 결과 없음")
except Exception as e:
    print(f"검색 오류: {e}")

=== 내용 확인 ======================================================================================================================================
page_content='id: 59647809, type: 웹소설, platform: 카카오페이지, title: 화산천마, status: 월~금 연재, thumbnail: https://page-images.kakaoentcdn.com/download/resource?kid=vPpRR/hAHIq1vh21/tedW3p9mEjLVCGyeMXNXBk&filename=o1/dims/resize/384, genre: 무협, views: 52453000, rating: 9.2, like: -, description: 천마신교의 교주이자, 천하제일인이라 불리는 절대천마. 영생불멸군림대법으로 인해 정파에서 깨어나고 마는데……. "뭐? 화산파? 이건 또 무슨 개소리야?" 갑자기 화산파의 제자가 된 마도 대종사의 이야기. 몰락의 길을 걷던 화산파가 다시 비상하기 시작한다., keyword: ['퓨전', '환생', '먼치킨', '고인물', '천마', '도사', '정파', '검', '화산'], author: 사야객, illustrator: -, original: -, age_rating: 전체이용가, price: ['1일 기다리면 무료', '', '100캐시']' metadata={'age_rating': '전체이용가', 'author': '사야객', 'description': '천마신교의 교주이자, 천하제일인이라 불리는 절대천마. 영생불멸군림대법으로 인해 정파에서 깨어나고 마는데……. "뭐? 화산파? 이건 또 무슨 개소리야?" 갑자기 화산파의 제자가 된 마도 대종사의 이야기. 몰락의 길을 걷던 화산파가 다시 비상하기 시작한다.', 'episode': 777, 'genre': '무협', 'id': 59647809, 'illu